In [3]:
!pip install transformers gradio

In [4]:
import torch
import pandas as pd
import re
import matplotlib.pyplot as plt
import gradio as gr
from torch import nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from bs4 import BeautifulSoup
from google.colab import drive
from transformers import GPT2Tokenizer, GPT2LMHeadModel, GPT2Config, StoppingCriteria, StoppingCriteriaList

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2-medium", bos_token='<|startoftext|>', eos_token='<|endoftext|>', pad_token='<|pad|>')

tokenizer.padding_side = 'right'  # Ensure padding is on the right
tokenizer.truncation_side = 'left'  # Truncate from the left

configuration = GPT2Config.from_pretrained('openai-community/gpt2-medium', output_hidden_states=False)

model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2-medium", config=configuration)
model.resize_token_embeddings(len(tokenizer))

# Path where you saved the weights
save_path = "/content/drive/MyDrive/Colab Notebooks/GPT2/fine-tuned_GPT2_work_arrangements_model.pth"

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the state_dict from disk
state_dict = torch.load(save_path, map_location=device)

# Populate model parameters
model.load_state_dict(state_dict)

# Send to device & set to eval mode
model.to(device)
model.eval()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50259, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50259, bias=False)
)

In [7]:
def cleanText(text):
    if pd.isnull(text):
        return ""
    soup = BeautifulSoup(text, "html.parser")
    cleaned_text = soup.get_text(separator=" ", strip=True)

    # This pattern matches anything that is not a letter, digit, space, or specified punctuation
    pattern = r'[^a-zA-Z0-9\s!?,.:();&$€£¥₹/\|-]'
    # Use re.sub() to replace unwanted characters with an empty string
    cleaned_text = re.sub(pattern, '', cleaned_text)
    return cleaned_text

In [8]:
# Define a custom stopping criterion that stops generation when a specific token is produced.
class StopOnToken(StoppingCriteria):
    def __init__(self, token_id):
        self.token_id = token_id

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # Check if the last token in the sequence is the stop token.
        # input_ids shape: [batch_size, sequence_length]
        if input_ids[0, -1] == self.token_id:
            return True
        return False

# Get the token id for the closing parenthesis: ')'
stop_token_id = tokenizer.encode(")")[0]

# Create a stopping criteria list with our custom criterion
stopping_criteria = StoppingCriteriaList([StopOnToken(stop_token_id)])

In [12]:
def predict(job_ad: str) -> str:
    prompt = f"Job Ad: ({job_ad.strip()}) Work Arrangements: ("
    prompt = cleanText(prompt)
    tokenized_input = tokenizer.encode(prompt, return_tensors="pt")
    tokenized_input = tokenized_input[:, -1000:]

    tokenized_input = tokenized_input.to(device)
    attention_mask = torch.ones(tokenized_input.shape, device=device)

    outputs = model.generate(input_ids=tokenized_input,
                  attention_mask=attention_mask,
                  pad_token_id=tokenizer.pad_token_id,
                  do_sample=True,
                  top_k=20,
                  max_new_tokens=5,
                  top_p=0.95,
                  num_return_sequences=1,
                  stopping_criteria=stopping_criteria)

    generated_tokens = outputs[0, tokenized_input.size(1)-1:]

    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Use a regular expression to capture the text between the first "(" and the first ")".
    match = re.search(r'\((.*?)\)', generated_text)
    if match:
        # This will capture the text inside the parentheses.
        extracted_text = match.group(1)
    else:
        extracted_text = generated_text[2:]  # Fallback in case no matching pattern is found
    return extracted_text

In [13]:
demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type your job ad here: "),
    outputs=gr.Textbox(label="Generated Work Arrangements"),
    title="My GPT-2_Work_Arrangements Demo",
    description="Enter a job ad and GPT-2_Work_Arrangements will output the work arrangement.",
)

In [14]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://04777e575fb88cf9ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
